# Experiment 3: Loss Function --- Focal Loss

## Rationale

Focal Loss down-weights easy samples (Trouser, Sandal, Sneaker) and amplifies gradient from the hard upper-body decision boundary, shrinking the confusion sink without the zero-sum trade-off observed in E2.

**Single variable changed**: `CrossEntropyLoss` rarr `FocalLoss(gamma=2.0)`
**Held constant**: architecture (DiagnosticCNN), data (no augmentation), optimizer (Adam lr=0.001), epochs (15)

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Import Libraries | Load PyTorch, define FocalLoss, detect device | --- |
| 2 | Load Dataset | FashionMNIST with train/test split | `src/data_utils.py` |
| 3 | Define DiagnosticCNN | Identical architecture to E1 baseline | --- |
| 4 | Train with FocalLoss | Train 15 epochs with FocalLoss(gamma=2.0) | `src/train_utils.py` |
| 5 | Evaluate Model | Per-class TPR, Precision, confusion matrix | `src/eval_utils.py` |
| 6 | ROC & PR Curves | ROC-AUC and PR-AUC scores | `src/eval_utils.py`, `src/vis_utils.py` |
| 7 | Compare with E1 | Side-by-side metrics vs DiagnosticCNN baseline | `src/eval_utils.py` |
| 8 | Save Outputs | Save metrics to outputs/error_analysis/focal_loss/ | --- |

---


In [ ]:
import os, sys
# Detect project root: look for src/ directory in CWD or parents
def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import os, torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim

import numpy as np, matplotlib.pyplot as plt

from src.data_utils import get_fashionmnist_transforms, load_fashionmnist, get_dataloaders

from src.train_utils import train_one_epoch

from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores)

OUT_DIR = '../outputs/error_analysis/focal_loss'
os.makedirs(OUT_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")

if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'

print(f"Device: {device}")

PyTorch: 2.13.0+cu130
Device: cuda


## Dataset — identical to E1

In [2]:
transform = get_fashionmnist_transforms()
train_dataset, test_dataset = load_fashionmnist(transform)
class_names = train_dataset.classes
train_loader, test_loader = get_dataloaders(train_dataset, test_dataset, batch_size=64)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 938, Test batches: 157


## Architecture — identical to E1 DiagnosticCNN

Reverted from E2's wider+residual. Exact same 3-block CNN (32→64→128, ReLU, Dropout 0.3).

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)

        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def get_features(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.global_pool(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        x = self.get_features(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = DiagnosticCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"DiagnosticCNN params: {total_params:,}")

DiagnosticCNN params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Loss Function — single variable change

`CrossEntropyLoss` → `FocalLoss(γ=2.0)`.

Focal Loss: `FL(p_t) = −α_t · (1 − p_t)^γ · log(p_t)`

- `p_t` = predicted probability for the true class
- `γ = 2.0` — modulating factor that down-weights easy samples (p_t → 1.0) and up-weights hard samples (p_t → 0.0)
- `α = None` — no class weighting (pure γ effect)

In [4]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


criterion = FocalLoss(gamma=2.0)
print(f"Loss: FocalLoss(γ=2.0)")

# Quick sanity: perfect prediction should have near-zero loss
logits_perfect = torch.tensor([[100.0, 0.0], [0.0, 100.0]])
targets_perfect = torch.tensor([0, 1])
loss_perfect = criterion(logits_perfect, targets_perfect)
print(f"  Perfect pred loss: {loss_perfect.item():.6f} (should be ~0)")

logits_uncertain = torch.tensor([[1.0, 1.0], [1.0, 1.0]])
loss_uncertain = criterion(logits_uncertain, targets_perfect)
print(f"  Uncertain pred loss: {loss_uncertain.item():.6f}")
ratio = loss_uncertain.item() / (loss_perfect.item() + 1e-8)
print(f"  Hard/Easy ratio: {ratio:.1f}x (Focal increases focus on uncertain samples)")

Loss: FocalLoss(γ=2.0)
  Perfect pred loss: 0.000000 (should be ~0)
  Uncertain pred loss: 0.173287
  Hard/Easy ratio: 17328679.6x (Focal increases focus on uncertain samples)


## Training — identical hyperparameters to E1

In [5]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 15

train_losses = []
model.train()
for epoch in range(num_epochs):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')
print(f"Losses saved.")

Epoch [1/15], Loss: 0.2431
Epoch [2/15], Loss: 0.1385
Epoch [3/15], Loss: 0.1155
Epoch [4/15], Loss: 0.1030
Epoch [5/15], Loss: 0.0912
Epoch [6/15], Loss: 0.0827
Epoch [7/15], Loss: 0.0763
Epoch [8/15], Loss: 0.0684
Epoch [9/15], Loss: 0.0640
Epoch [10/15], Loss: 0.0584
Epoch [11/15], Loss: 0.0539
Epoch [12/15], Loss: 0.0487
Epoch [13/15], Loss: 0.0451
Epoch [14/15], Loss: 0.0406
Epoch [15/15], Loss: 0.0372
Losses saved.


## Evaluation — identical pipeline to E1

In [6]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='FocalCNN')
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='FocalCNN')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='FocalCNN')

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write('-' * 55 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')

cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names:
        f.write(f'{name:>15}')
    f.write('\n')
    for i in range(len(class_names)):
        f.write(f'{class_names[i]:>15}')
        for j in range(len(class_names)):
            f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        true_name = class_names[c]
        total_errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {true_name}  (errors: {total_errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0:
                continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f"\nAll results saved to {OUT_DIR}/")

  Test Accuracy: 92.40%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8810     0.0151     0.8663
  Trouser             0.9880     0.0011     0.9900
  Pullover            0.8960     0.0111     0.8996
  Dress               0.9330     0.0090     0.9201
  Coat                0.9310     0.0200     0.8380
  Sandal              0.9610     0.0008     0.9928
  Shirt               0.7130     0.0150     0.8408
  Sneaker             0.9820     0.0061     0.9470
  Bag                 0.9930     0.0029     0.9745
  Ankle boot          0.9620     0.0033     0.9698

All results saved to ../outputs/error_analysis/focal_loss/


## Delta vs E1 Baseline

In [7]:
def load_e1_metrics(path):
    data = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5 and parts[0] != 'Class' and '-' not in line[:5]:
                cls, roc, pr, tpr, prec = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                data[cls] = {'roc_auc': roc, 'pr_auc': pr, 'tpr': tpr, 'precision': prec}
    return data

e1 = load_e1_metrics('../outputs/error_analysis/metrics_summary.txt')

print(f'{"Class":<15} {"E1 TPR":>8} {"E3 TPR":>8} {"Δ TPR":>8} {"E1 Prec":>8} {"E3 Prec":>8} {"Δ Prec":>8}')
print('-' * 63)
for name in class_names:
    if name in e1:
        e1_tpr = e1[name]['tpr']
        e1_prec = e1[name]['precision']
        e3_tpr = per_class[name]['TPR']
        e3_prec = per_class[name]['Precision']
        e3_pr = pr_scores[f'class_{class_names.index(name)}']
        print(f'{name:<15} {e1_tpr:>8.3f} {e3_tpr:>8.3f} {e3_tpr - e1_tpr:>+8.3f} {e1_prec:>8.3f} {e3_prec:>8.3f} {e3_prec - e1_prec:>+8.3f}')

print(f'\nAccuracy:  E1=92.50%  E3={accuracy:.2f}%  Δ={accuracy - 92.50:+.2f}%')
print(f'Macro PR:   E1=0.9712  E3={pr_scores["macro"]:.4f}  Δ={pr_scores["macro"] - 0.9712:+.4f}')

Class             E1 TPR   E3 TPR    Δ TPR  E1 Prec  E3 Prec   Δ Prec
---------------------------------------------------------------
Trouser            0.988    0.988   +0.000    0.990    0.990   -0.000
Pullover           0.890    0.896   +0.006    0.896    0.900   +0.003
Dress              0.907    0.933   +0.026    0.940    0.920   -0.020
Coat               0.870    0.931   +0.061    0.931    0.838   -0.094
Sandal             0.978    0.961   -0.017    0.990    0.993   +0.003
Shirt              0.847    0.713   -0.134    0.723    0.841   +0.118
Sneaker            0.991    0.982   -0.009    0.946    0.947   +0.001
Bag                0.986    0.993   +0.007    0.982    0.974   -0.008

Accuracy:  E1=92.50%  E3=92.40%  Δ=-0.10%
Macro PR:   E1=0.9712  E3=0.9706  Δ=-0.0006


## Results saved to `outputs/error_analysis/focal_loss/`